# TruthLens — Fine-tuning DistilBERT for Binary Fake News Detection


## 1. Imports and configuration

In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GB")

## 2. Load and prepare the LIAR dataset

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/tfs4/liar_dataset/master/"

COLUMNS = [
    "id", "label", "statement", "subject", "speaker", "job",
    "state", "party", "barely_true_count", "false_count", "half_true_count",
    "mostly_true_count", "pants_fire_count", "context",
]

LIAR_LABELS = [
    "false",
    "half-true",
    "mostly-true",
    "true",
    "barely-true",
    "pants-fire",
]

BINARY_LABEL_MAP = {
    "pants-fire": 0,
    "false": 0,
    "barely-true": 0,
    "half-true": 1,
    "mostly-true": 1,
    "true": 1,
}

LABEL_NAMES = ["FAKE", "REAL"]


def load_split(filename):
    df = pd.read_csv(f"{BASE_URL}{filename}", sep="\t", header=None, names=COLUMNS)

    df = df[["statement", "label"]].dropna()
    df["label"] = df["label"].str.strip().str.lower()
    df = df[df["label"].isin(LIAR_LABELS)]

    df["label"] = df["label"].map(BINARY_LABEL_MAP).astype(int)
    return Dataset.from_pandas(df, preserve_index=False)


dataset = DatasetDict({
    "train": load_split("train.tsv"),
    "validation": load_split("valid.tsv"),
    "test": load_split("test.tsv"),
})

print(dataset)
print("\nSample:")
print(dataset["train"][0])

In [ ]:
train_df = dataset["train"].to_pandas()
counts = train_df["label"].value_counts().sort_index()

print("Training label distribution:")
for label_id, count in counts.items():
    print(f"{LABEL_NAMES[label_id]}: {count}")

os.makedirs("./outputs", exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(LABEL_NAMES, counts.values, width=0.45)
ax.set_title("LIAR train set label distribution")
ax.set_ylabel("Count")

for idx, value in enumerate(counts.values):
    ax.text(idx, value + 30, str(value), ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("./outputs/label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Tokenize statements

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)


def tokenize(batch):
    return tokenizer(
        batch["statement"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized = dataset.map(tokenize, batched=True, remove_columns=["statement"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

print(tokenized)

## 4. Configure DistilBERT

In [ ]:
id2label = {0: "FAKE", 1: "REAL"}
label2id = {"FAKE": 0, "REAL": 1}

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

## 5. Train the model

In [ ]:
if DEVICE.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    batch_size = 8 if vram_gb < 5 else 16
    grad_accumulation = 4 if vram_gb < 5 else 2
    use_fp16 = True
else:
    batch_size = 8
    grad_accumulation = 2
    use_fp16 = False

print(f"Batch size: {batch_size}")
print(f"Gradient accumulation: {grad_accumulation}")
print(f"Effective batch size: {batch_size * grad_accumulation}")

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    gradient_accumulation_steps=grad_accumulation,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    fp16=use_fp16,
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
train_result = trainer.train()

print("Training complete")
print(f"Runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")

## 6. Evaluate the model

In [ ]:
validation_metrics = trainer.evaluate()

print("Validation metrics:")
for key, value in validation_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
test_output = trainer.predict(tokenized["test"])
y_pred = np.argmax(test_output.predictions, axis=-1)
y_true = test_output.label_ids

test_accuracy = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="weighted")

report = classification_report(y_true, y_pred, target_names=LABEL_NAMES)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test F1: {test_f1:.4f}")
print()
print(report)

os.makedirs("./outputs", exist_ok=True)

with open("./outputs/classification_report.txt", "w", encoding="utf-8") as file:
    file.write(f"Test accuracy: {test_accuracy:.4f}\n")
    file.write(f"Test F1: {test_f1:.4f}\n\n")
    file.write(report)

print("Saved: outputs/classification_report.txt")

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred)
conf_matrix_norm = conf_matrix.astype("float") / conf_matrix.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, fmt, title in zip(
    axes,
    [conf_matrix, conf_matrix_norm],
    ["d", ".2f"],
    ["Confusion matrix", "Normalized confusion matrix"],
):
    sns.heatmap(
        data,
        annot=True,
        fmt=fmt,
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES,
        ax=ax,
        linewidths=0.5,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)

plt.tight_layout()
plt.savefig("./outputs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: outputs/confusion_matrix.png")

In [ ]:
history = trainer.state.log_history

train_steps = [entry["step"] for entry in history if "loss" in entry]
train_losses = [entry["loss"] for entry in history if "loss" in entry]

eval_steps = [entry["step"] for entry in history if "eval_loss" in entry]
eval_losses = [entry["eval_loss"] for entry in history if "eval_loss" in entry]

eval_epochs = [entry["epoch"] for entry in history if "eval_accuracy" in entry]
eval_accuracies = [entry["eval_accuracy"] for entry in history if "eval_accuracy" in entry]

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))

ax_loss.plot(train_steps, train_losses, label="Train loss")
ax_loss.plot(eval_steps, eval_losses, "o--", label="Validation loss")
ax_loss.set_xlabel("Step")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Loss curve")
ax_loss.legend()

ax_acc.plot(eval_epochs, eval_accuracies, "o-")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title("Validation accuracy")

plt.tight_layout()
plt.savefig("./outputs/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: outputs/training_curves.png")

## 7. Explain one prediction with SHAP

In [ ]:
import shap

model.to(DEVICE)
model.eval()


def predict_proba(texts):
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**encoded).logits

    return torch.softmax(logits, dim=-1).cpu().numpy()


test_texts = dataset["test"]["statement"]
test_labels = dataset["test"]["label"]

sample_text = next(text for text, label in zip(test_texts, test_labels) if label == 0)

print("Statement:")
print(sample_text)

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_proba, masker, max_evals=300)
shap_values = explainer([sample_text])

prediction = int(np.argmax(predict_proba([sample_text])))
print(f"Predicted: {LABEL_NAMES[prediction]}")

shap.plots.text(shap_values[0, :, 1])

## 8. Save the model

In [ ]:
SAVE_DIR = "./model/saved_distilbert"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")
print(os.listdir(SAVE_DIR))

## 9. Quick inference check

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if DEVICE.type == "cuda" else -1,
    top_k=None,
)

test_statements = [
    "The government is hiding evidence of alien contact at Area 51.",
    "Exercise reduces the risk of cardiovascular disease.",
    "Unemployment reached its lowest point in 50 years last quarter.",
    "Vaccines contain microchips that track your location.",
]

for statement in test_statements:
    result = classifier(statement)[0]
    top_result = max(result, key=lambda item: item["score"])

    print(f"Statement: {statement}")
    print(f"Prediction: {top_result['label']} ({top_result['score']:.1%})")
    print()